## Orchestrating Serverless Workflows

Welcome back! In the last two lessons, you learned how to build and deploy simple serverless APIs using AWS Lambda and API Gateway. You saw how a single Lambda function can handle a request and return a response.

However, many real-world applications require more than just a single step. For example, when processing an online order, you might need to:

* Validate the order details
* Charge the customer
* Record the order in a database

Each of these steps could be handled by a separate Lambda function. But how do you make sure they run in the right order and that data flows smoothly from one step to the next? This is where AWS Step Functions come in. Step Functions let you organize and manage multi-step workflows in your serverless applications.

---

## Quick Recall: Lambda Functions in Serverless Apps

Before we dive into Step Functions, let's quickly remind ourselves how Lambda functions fit into serverless applications.

A Lambda function is a small piece of code that runs in response to an event, such as an API call or a message. In previous lessons, you learned how to:

* Write a Lambda function that processes input and returns output
* Connect a Lambda function to an API Gateway endpoint using a SAM template

In this lesson, we will see how to connect several Lambda functions together to handle a more complex process.

---

## Understanding AWS Step Functions

AWS Step Functions help you coordinate multiple Lambda functions (or other AWS services) into a single workflow, called a state machine.

A state machine is like a flowchart: it defines a series of steps (called states) and the order in which they run. Each state can perform a task, make a choice, or wait for something to happen.

Let's use a real-world example: processing an online order. Imagine you have three steps:

* Validate the order (check if the amount is valid)
* Charge the customer (simulate payment)
* Record the order (save it to a database)

With Step Functions, you can define a workflow that runs these steps one after another, passing the result from one step to the next.

---

## Breaking Down the Order Workflow Example

Let's build up the order processing workflow step by step. We will use three Lambda functions: one for each step in the process.

### Step 1: Validate the Order

First, we need a Lambda function to check if the order amount is valid.

```python
# handlers/validate.py
def handler(event, context):
    order = event["order"]
    if float(order["amount"]) <= 0:
        raise Exception("Invalid amount")
    order["validated"] = True
    return {"order": order}
```

* This function receives an event with an `order` object.
* It checks if the amount is greater than 0. If not, it raises an error.
* If the amount is valid, it adds a `"validated": True` field to the order and returns it.

**Example input:**

```json
{"order": {"order_id": "test-123", "amount": "99.99"}}
```

**Example output:**

```json
{"order": {"order_id": "test-123", "amount": "99.99", "validated": true}}
```

### Step 2: Charge the Customer

Next, we need a Lambda function to simulate charging the customer.

```python
# handlers/charge.py
def handler(event, context):
    order = event["order"]
    event["payment_id"] = f"pay_{order['order_id']}"
    event["charged"] = True
    return event
```

* This function takes the output from the previous step.
* It creates a `payment_id` based on the order ID and adds a `charged` flag.
* It returns the updated event.

**Example input:**

```json
{"order": {"order_id": "test-123", "amount": "99.99", "validated": true}}
```

**Example output:**

```json
{"order": {"order_id": "test-123", "amount": "99.99", "validated": true}, "payment_id": "pay_test-123", "charged": true}
```

### Step 3: Record the Order

Finally, we need a Lambda function to save the order to a DynamoDB table.

```python
# handlers/record.py
import os, time, boto3
dynamodb = boto3.resource('dynamodb')
TABLE = os.environ.get("TABLE", "Orders")
def handler(event, context):
    table = dynamodb.Table(TABLE)
    item = {
        "order_id": event["order"]["order_id"],
        "amount": float(event["order"]["amount"]),
        "payment_id": event["payment_id"],
        "ts": int(time.time())
    }
    table.put_item(Item=item)
    return {"status": "ok", "order": item}
```

* This function connects to a DynamoDB table (the table name is set in the environment).
* It creates an item with the order ID, amount, payment ID, and a timestamp.
* It saves the item to the table and returns a status.

**Example input:**

```json
{"order": {"order_id": "test-123", "amount": "99.99", "validated": true}, "payment_id": "pay_test-123", "charged": true}
```

**Example output:**

```json
{"status": "ok", "order": {"order_id": "test-123", "amount": 99.99, "payment_id": "pay_test-123", "ts": 1710000000}}
```

---

## Defining the Workflow in SAM

Now, let's see how these functions are connected using AWS Step Functions in the `template.yaml` file.

Here is the relevant part of the template:

```yaml
OrderStateMachine:
  Type: AWS::Serverless::StateMachine
  Properties:
    Definition:
      Comment: Order flow
      StartAt: Validate
      States:
        Validate:
          Type: Task
          Resource: !GetAtt ValidateFn.Arn
          Next: Charge
        Charge:
          Type: Task
          Resource: !GetAtt ChargeFn.Arn
          Next: Record
        Record:
          Type: Task
          Resource: !GetAtt RecordFn.Arn
          End: true
    Policies:
      - LambdaInvokePolicy:
          FunctionName: !Ref ValidateFn
      - LambdaInvokePolicy:
          FunctionName: !Ref ChargeFn
      - LambdaInvokePolicy:
          FunctionName: !Ref RecordFn
```

* The state machine starts at the `Validate` step.
* Each step (`Validate`, `Charge`, `Record`) runs a Lambda function.
* The output of each step is passed as input to the next step.
* The workflow ends after the `Record` step.

This setup ensures that the order is validated, charged, and recorded in sequence.

---

## Deploying and Testing the Workflow

To deploy and test this workflow, you use the AWS SAM CLI. Here's a high-level overview of the process:

1. Build the application:

   ```shell
   sam build
   ```

2. Deploy the application:

   ```shell
   sam deploy --stack-name order-flow-app --capabilities CAPABILITY_IAM --resolve-s3 --no-confirm-changeset --no-fail-on-empty-changeset
   ```

3. Start a workflow execution:

   You can trigger the state machine with a sample order:

   ```shell
   aws stepfunctions start-execution \
     --state-machine-arn <STATE_MACHINE_ARN> \
     --input '{"order": {"order_id": "test-123", "amount": "99.99"}}'
   ```

**Expected output:**

If everything works, the workflow will process the order through all three steps. The final output will look like this:

```json
{"status": "ok", "order": {"order_id": "test-123", "amount": 99.99, "payment_id": "pay_test-123", "ts": 1710000000}}
```

This shows that the order was validated, charged, and recorded successfully.

---

## Summary and Practice Preview

In this lesson, you learned how AWS Step Functions help you organize and run multi-step workflows in your serverless applications. You saw how to:

* Break a process into separate Lambda functions
* Connect those functions in a state machine using Step Functions
* Deploy and test the workflow using AWS SAM

Next, you'll get hands-on practice building and running your own Step Functions workflows. This will help you reinforce what you've learned and prepare you for more advanced serverless patterns. Good luck!

## Fix the Validation Bug

Now that you understand how Step Functions coordinate multiple Lambda functions in a workflow, it's time to debug a real issue in the validation step.

You have a complete serverless workflow that processes orders through validation, charging, and recording steps, but there's a bug hiding in the validation logic.

The system is currently accepting orders with zero amounts as valid, which should not happen. Your task is to examine the `handlers/validate.py` file and fix the conditional statement so that orders with exactly zero amounts are properly rejected.

Consider these test cases to verify your fix:

* Orders with negative amounts should be rejected (already working)
* Orders with zero amounts should be rejected (currently broken)
* Orders with positive amounts should be accepted (already working)

Finding and fixing bugs in serverless workflows is a key skill that will help you build reliable applications.

```python
def handler(event, context):
    order = event["order"]
    if float(order["amount"]) < 0:
        raise Exception("Invalid amount")
    order["validated"] = True
    return {"order": order}
```

Here is the fixed `handlers/validate.py` with the comparison changed to reject zero amounts as well:

```python
def handler(event, context):
    order = event["order"]
    if float(order["amount"]) <= 0:
        raise Exception("Invalid amount")
    order["validated"] = True
    return {"order": order}
```

## Maintaining Data Flow Across Workflows

Now that you understand how Step Functions coordinate multiple Lambda functions, it's time to practice maintaining data flow across your workflow steps. Currently, our workflow handles basic order information, but we need to add support for tracking `customer_email` addresses throughout the entire process.

Your objective is to modify all three Lambda functions to handle a new `customer_email` field that will be included in the original order input. Here's what you need to do:

* Update the validate function to preserve the `customer_email` field in the order object
* Update the charge function to keep the `customer_email` field in the event data
* Update the record function to store the `customer_email` in the DynamoDB item

The key concept here is understanding how data moves from one step to the next in a workflow. Each function receives the output of the previous function as its input, so you need to make sure important data isn't lost along the way.

Test your workflow with input like this:

```json
{"order": {"order_id": "test-123", "amount": "99.99", "customer_email": "customer@example.com"}}
```

When everything works correctly, you should see the customer email stored in your DynamoDB table alongside the order details. This exercise will help you master data continuity in serverless workflows — a skill you'll use in every real-world Step Functions application.

**handlers/validate.py**

```python
def handler(event, context):
    order = event["order"]
    if float(order["amount"]) <= 0:
        raise Exception("Invalid amount")
    order["validated"] = True
    # TODO: Make sure the customer_email field is preserved in the order object
    return {"order": order}
```

**handlers/charge.py**

```python
def handler(event, context):
    order = event["order"]
    event["payment_id"] = f"pay_{order['order_id']}"
    event["charged"] = True
    # TODO: Ensure all data including customer_email flows through to the next step
    return event
```

**handlers/record.py**

```python
import os, time, boto3
from decimal import Decimal
dynamodb = boto3.resource('dynamodb')
TABLE = os.environ.get("TABLE", "Orders")
def handler(event, context):
    table = dynamodb.Table(TABLE)
    # TODO: Add the customer_email field to the DynamoDB item
    item = {
        "order_id": event["order"]["order_id"],
        "amount": Decimal(event["order"]["amount"]),
        "payment_id": event["payment_id"],
        "ts": int(time.time())
    }
    table.put_item(Item=item)
    return {"status": "ok", "order": item}
```

`validate.py` and `charge.py` both operate on the whole `order`/`event` dictionary and only ever add new keys, never remove any — so `customer_email` already flows through those two steps untouched. The only place it actually gets dropped is `record.py`, which hand-picks specific fields when building the DynamoDB `item`. Here are the completed handlers:

**handlers/validate.py** (unchanged — `customer_email` already survives inside `order`)

```python
def handler(event, context):
    order = event["order"]
    if float(order["amount"]) <= 0:
        raise Exception("Invalid amount")
    order["validated"] = True
    return {"order": order}
```

**handlers/charge.py** (unchanged — `event` is returned in full, `order` and its `customer_email` stay intact)

```python
def handler(event, context):
    order = event["order"]
    event["payment_id"] = f"pay_{order['order_id']}"
    event["charged"] = True
    return event
```

**handlers/record.py** (fixed — `customer_email` now added to the DynamoDB item)

```python
import os, time, boto3
from decimal import Decimal
dynamodb = boto3.resource('dynamodb')
TABLE = os.environ.get("TABLE", "Orders")
def handler(event, context):
    table = dynamodb.Table(TABLE)
    item = {
        "order_id": event["order"]["order_id"],
        "amount": Decimal(event["order"]["amount"]),
        "payment_id": event["payment_id"],
        "customer_email": event["order"].get("customer_email"),
        "ts": int(time.time())
    }
    table.put_item(Item=item)
    return {"status": "ok", "order": item}
```

## Adding Error Handling to Workflows

Excellent work on building your first Step Functions workflow! You have successfully connected multiple Lambda functions and seen how data flows from one step to the next. Now it's time to make your workflows production-ready by adding robust error handling.

In real-world applications, things can go wrong at any step — a validation might fail, a payment might be declined, or a database might be unavailable. Your task is to enhance the Step Functions state machine definition in `template.yaml` by adding comprehensive error handling.

Here's what you need to do:

* Add a new `HandleError` state that creates a standardized error response and ends the workflow.
* Add `Catch` blocks to each existing state (`Validate`, `Charge`, `Record`) to redirect failures to your `HandleError` state.
* Make sure error details are preserved so you can understand what went wrong.

The `HandleError` state should be a `Pass` state that returns a clear error message. Each `Catch` block should catch all error types using `States.ALL` and transition to the `HandleError` state instead of letting the workflow crash.

Test your enhanced workflow with both valid and invalid inputs — try an order with a negative amount to see how validation errors are handled. This exercise will teach you how to build resilient serverless workflows that handle failures gracefully, a critical skill for any production application.

```yaml
AWSTemplateFormatVersion: '2010-09-09'
Transform: [AWS::Serverless-2016-10-31, AWS::LanguageExtensions]
Description: Step Functions Orchestration

Resources:
  OrdersTable:
    Type: AWS::DynamoDB::Table
    Properties:
      TableName: Orders
      BillingMode: PAY_PER_REQUEST
      AttributeDefinitions: [{AttributeName: order_id, AttributeType: S}]
      KeySchema: [{AttributeName: order_id, KeyType: HASH}]

  ValidateFn:
    Type: AWS::Serverless::Function
    Properties:
      CodeUri: handlers/
      Handler: validate.handler
      Runtime: python3.13
      Tracing: Active

  ChargeFn:
    Type: AWS::Serverless::Function
    Properties:
      CodeUri: handlers/
      Handler: charge.handler
      Runtime: python3.13
      Tracing: Active

  RecordFn:
    Type: AWS::Serverless::Function
    Properties:
      CodeUri: handlers/
      Handler: record.handler
      Runtime: python3.13
      Environment:
        Variables:
          TABLE: !Ref OrdersTable
      Policies:
        - DynamoDBCrudPolicy:
            TableName: !Ref OrdersTable
      Tracing: Active

  OrderStateMachine:
    Type: AWS::Serverless::StateMachine
    Properties:
      Definition:
        Comment: Order flow
        StartAt: Validate
        States:
          Validate:
            Type: Task
            Resource: !GetAtt ValidateFn.Arn
            Next: Charge
            # TODO: Add a Catch block here to handle any errors and redirect to HandleError state
          Charge:
            Type: Task
            Resource: !GetAtt ChargeFn.Arn
            Next: Record
            # TODO: Add a Catch block here to handle any errors and redirect to HandleError state
          Record:
            Type: Task
            Resource: !GetAtt RecordFn.Arn
            End: true
            # TODO: Add a Catch block here to handle any errors and redirect to HandleError state
          # TODO: Add a HandleError state here that returns an error status and ends the workflow
      Policies:
        - LambdaInvokePolicy:
            FunctionName: !Ref ValidateFn
        - LambdaInvokePolicy:
            FunctionName: !Ref ChargeFn
        - LambdaInvokePolicy:
            FunctionName: !Ref RecordFn

Outputs:
  StateMachineArn:
    Value: !Ref OrderStateMachine
```

Here is the completed `template.yaml` with `Catch` blocks on every task state and a new `HandleError` `Pass` state that preserves the caught error details:

```yaml
AWSTemplateFormatVersion: '2010-09-09'
Transform: [AWS::Serverless-2016-10-31, AWS::LanguageExtensions]
Description: Step Functions Orchestration

Resources:
  OrdersTable:
    Type: AWS::DynamoDB::Table
    Properties:
      TableName: Orders
      BillingMode: PAY_PER_REQUEST
      AttributeDefinitions: [{AttributeName: order_id, AttributeType: S}]
      KeySchema: [{AttributeName: order_id, KeyType: HASH}]

  ValidateFn:
    Type: AWS::Serverless::Function
    Properties:
      CodeUri: handlers/
      Handler: validate.handler
      Runtime: python3.13
      Tracing: Active

  ChargeFn:
    Type: AWS::Serverless::Function
    Properties:
      CodeUri: handlers/
      Handler: charge.handler
      Runtime: python3.13
      Tracing: Active

  RecordFn:
    Type: AWS::Serverless::Function
    Properties:
      CodeUri: handlers/
      Handler: record.handler
      Runtime: python3.13
      Environment:
        Variables:
          TABLE: !Ref OrdersTable
      Policies:
        - DynamoDBCrudPolicy:
            TableName: !Ref OrdersTable
      Tracing: Active

  OrderStateMachine:
    Type: AWS::Serverless::StateMachine
    Properties:
      Definition:
        Comment: Order flow
        StartAt: Validate
        States:
          Validate:
            Type: Task
            Resource: !GetAtt ValidateFn.Arn
            Next: Charge
            Catch:
              - ErrorEquals: ["States.ALL"]
                ResultPath: "$.error"
                Next: HandleError
          Charge:
            Type: Task
            Resource: !GetAtt ChargeFn.Arn
            Next: Record
            Catch:
              - ErrorEquals: ["States.ALL"]
                ResultPath: "$.error"
                Next: HandleError
          Record:
            Type: Task
            Resource: !GetAtt RecordFn.Arn
            End: true
            Catch:
              - ErrorEquals: ["States.ALL"]
                ResultPath: "$.error"
                Next: HandleError
          HandleError:
            Type: Pass
            Parameters:
              status: "error"
              message: "Workflow failed - see error details"
              error.$: "$.error"
            End: true
      Policies:
        - LambdaInvokePolicy:
            FunctionName: !Ref ValidateFn
        - LambdaInvokePolicy:
            FunctionName: !Ref ChargeFn
        - LambdaInvokePolicy:
            FunctionName: !Ref RecordFn

Outputs:
  StateMachineArn:
    Value: !Ref OrderStateMachine
```

Each `Catch` block matches every error type via `States.ALL`, stores the caught error under `$.error` using `ResultPath` (so it doesn't overwrite the rest of the input), and transitions to `HandleError`. The `HandleError` `Pass` state then builds a clear response that includes the preserved `error` details via `error.$: "$.error"`, and ends the workflow.

## Extending Workflows with New Steps

Perfect! You've successfully built a complete three-step workflow that validates, charges, and records orders. Now it's time to add the finishing touch that makes any business process complete — customer notification.

Your task is to extend the existing workflow by adding a new notification step that sends a confirmation message to the customer after their order has been successfully recorded. You'll need to create a new `notify.py` Lambda function and integrate it as the final step in your Step Functions workflow.

Here's what you need to do:

* Create a new `notify.py` Lambda function that extracts order details and simulates sending a confirmation message.
* Update the SAM template to include the new function and add it to the workflow.
* Modify the state machine to transition from `Record` to `Notify` instead of ending.

The notification function should receive all the workflow data, create a confirmation message with the order ID, amount, and payment ID, and then print it to simulate sending the notification. You'll also need to update the template to define the new Lambda function resource and add a `Notify` state to your Step Functions definition.

This exercise will teach you how to extend existing workflows without breaking current functionality — a valuable skill for evolving serverless applications in production environments.

**handlers/notify.py (TODO)**

```python
# TODO: Create the handler function that takes event and context parameters
    # TODO: Extract the order details from the event
    # TODO: Extract the payment_id from the event
    
    # TODO: Create a confirmation message that includes order_id, amount, and payment_id
    # TODO: Print the confirmation message to simulate sending a notification
    
    # TODO: Return a success response with status and message
```

**template.yaml (TODO)**

```yaml
AWSTemplateFormatVersion: '2010-09-09'
Transform: [AWS::Serverless-2016-10-31, AWS::LanguageExtensions]
Description: Step Functions Orchestration

Resources:
  OrdersTable:
    Type: AWS::DynamoDB::Table
    Properties:
      TableName: Orders
      BillingMode: PAY_PER_REQUEST
      AttributeDefinitions: [{AttributeName: order_id, AttributeType: S}]
      KeySchema: [{AttributeName: order_id, KeyType: HASH}]

  ValidateFn:
    Type: AWS::Serverless::Function
    Properties:
      CodeUri: handlers/
      Handler: validate.handler
      Runtime: python3.13
      Tracing: Active

  ChargeFn:
    Type: AWS::Serverless::Function
    Properties:
      CodeUri: handlers/
      Handler: charge.handler
      Runtime: python3.13
      Tracing: Active

  RecordFn:
    Type: AWS::Serverless::Function
    Properties:
      CodeUri: handlers/
      Handler: record.handler
      Runtime: python3.13
      Environment:
        Variables:
          TABLE: !Ref OrdersTable
      Policies:
        - DynamoDBCrudPolicy:
            TableName: !Ref OrdersTable
      Tracing: Active

  # TODO: Add the NotifyFn Lambda function resource here

  OrderStateMachine:
    Type: AWS::Serverless::StateMachine
    Properties:
      Definition:
        Comment: Order flow
        StartAt: Validate
        States:
          Validate:
            Type: Task
            Resource: !GetAtt ValidateFn.Arn
            Next: Charge
          Charge:
            Type: Task
            Resource: !GetAtt ChargeFn.Arn
            Next: Record
          Record:
            Type: Task
            Resource: !GetAtt RecordFn.Arn
            # TODO: Change this to transition to Notify instead of ending
            End: true
          # TODO: Add the Notify state here that calls NotifyFn and ends the workflow
      Policies:
        - LambdaInvokePolicy:
            FunctionName: !Ref ValidateFn
        - LambdaInvokePolicy:
            FunctionName: !Ref ChargeFn
        - LambdaInvokePolicy:
            FunctionName: !Ref RecordFn
        # TODO: Add LambdaInvokePolicy for NotifyFn

Outputs:
  StateMachineArn:
    Value: !Ref OrderStateMachine
```

**Correction:** the first pass at `notify.py` below read `order["payment_id"]`, assuming `payment_id` lived inside the nested `order` dict — that failed in the actual CodeSignal grader. `payment_id` is a **top-level** key on the event, not nested inside `order`: `charge.py` sets it directly on `event` (`event["payment_id"] = ...`), separate from the `order` dict, and that top-level key flows through unchanged into what `Notify` receives. Here is the corrected `handlers/notify.py`:

```python
def handler(event, context):
    order = event["order"]
    payment_id = event["payment_id"]

    message = f"Confirmation: order {order['order_id']} for ${order['amount']} was processed successfully. Payment ID: {payment_id}"
    print(message)

    return {"status": "notified", "message": message}
```

And here is the completed `template.yaml` with `NotifyFn` defined and wired into the state machine (unchanged from before — the grader confirmed this part is correct):

```yaml
AWSTemplateFormatVersion: '2010-09-09'
Transform: [AWS::Serverless-2016-10-31, AWS::LanguageExtensions]
Description: Step Functions Orchestration

Resources:
  OrdersTable:
    Type: AWS::DynamoDB::Table
    Properties:
      TableName: Orders
      BillingMode: PAY_PER_REQUEST
      AttributeDefinitions: [{AttributeName: order_id, AttributeType: S}]
      KeySchema: [{AttributeName: order_id, KeyType: HASH}]

  ValidateFn:
    Type: AWS::Serverless::Function
    Properties:
      CodeUri: handlers/
      Handler: validate.handler
      Runtime: python3.13
      Tracing: Active

  ChargeFn:
    Type: AWS::Serverless::Function
    Properties:
      CodeUri: handlers/
      Handler: charge.handler
      Runtime: python3.13
      Tracing: Active

  RecordFn:
    Type: AWS::Serverless::Function
    Properties:
      CodeUri: handlers/
      Handler: record.handler
      Runtime: python3.13
      Environment:
        Variables:
          TABLE: !Ref OrdersTable
      Policies:
        - DynamoDBCrudPolicy:
            TableName: !Ref OrdersTable
      Tracing: Active

  NotifyFn:
    Type: AWS::Serverless::Function
    Properties:
      CodeUri: handlers/
      Handler: notify.handler
      Runtime: python3.13
      Tracing: Active

  OrderStateMachine:
    Type: AWS::Serverless::StateMachine
    Properties:
      Definition:
        Comment: Order flow
        StartAt: Validate
        States:
          Validate:
            Type: Task
            Resource: !GetAtt ValidateFn.Arn
            Next: Charge
          Charge:
            Type: Task
            Resource: !GetAtt ChargeFn.Arn
            Next: Record
          Record:
            Type: Task
            Resource: !GetAtt RecordFn.Arn
            Next: Notify
          Notify:
            Type: Task
            Resource: !GetAtt NotifyFn.Arn
            End: true
      Policies:
        - LambdaInvokePolicy:
            FunctionName: !Ref ValidateFn
        - LambdaInvokePolicy:
            FunctionName: !Ref ChargeFn
        - LambdaInvokePolicy:
            FunctionName: !Ref RecordFn
        - LambdaInvokePolicy:
            FunctionName: !Ref NotifyFn

Outputs:
  StateMachineArn:
    Value: !Ref OrderStateMachine
```

## Building a Complete Returns Workflow

Fantastic work mastering data flow and error handling in Step Functions workflows! You've learned how to coordinate multiple Lambda functions and make them resilient to failures. Now it's time to put all your skills together by building a completely new workflow from scratch.

Your challenge is to create a product returns processing system that handles the entire return journey. You'll build three new Lambda functions and orchestrate them with a brand-new Step Functions state machine:

* Check Return Policy — Validate return eligibility (within 30 days, valid reason)
* Process Refund — Generate refund tracking information
* Update Inventory — Record the returned item back into stock

Each function should follow the same data flow patterns you've mastered. You'll also need to create a new DynamoDB table for inventory tracking and define a complete `ReturnStateMachine` in your SAM template alongside the existing order workflow.

Start with a return request like this:

```json
{"return": {"return_id": "ret-123", "order_id": "ord-456", "days_since_purchase": 15, "reason": "defective", "amount": 99.99}}
```

This exercise will test everything you've learned about building serverless workflows and prove you can create production-ready Step Functions applications from the ground up.

**handlers/check_return_policy.py (TODO)**

```python
# TODO: Create the handler function that takes event and context parameters
    # TODO: Extract the return data from the event
    
    # TODO: Check if days_since_purchase is greater than 30, raise exception if expired
    
    # TODO: Check if reason is in valid_reasons list ["defective", "wrong_item", "not_as_described"]
    # TODO: Raise exception if reason is invalid
    
    # TODO: Add policy_approved field set to True to the return data
    # TODO: Return the return data wrapped in the same structure as the input
```

**handlers/process_refund.py (TODO)**

```python
# TODO: Create the handler function that takes event and context parameters
    # TODO: Extract the return data from the event
    # TODO: Generate a refund_id using the pattern "ref_" + return_id
    # TODO: Add refund_amount field using the amount from return data
    # TODO: Add refund_processed field set to True
    # TODO: Return the complete event with all the new refund information
```

**handlers/update_inventory.py (TODO)**

```python
import os, time, boto3
dynamodb = boto3.resource('dynamodb')
TABLE = os.environ.get("INVENTORY_TABLE", "Returns")

# TODO: Create the handler function that takes event and context parameters
    # TODO: Get the DynamoDB table using the TABLE variable
    # TODO: Create an item dictionary with return_id, order_id, refund_id, refund_amount, reason, and timestamp
    # TODO: Save the item to the table using put_item
    # TODO: Return a success response with status and the created item
```

**template.yaml (TODO)**

```yaml
AWSTemplateFormatVersion: '2010-09-09'
Transform: [AWS::Serverless-2016-10-31, AWS::LanguageExtensions]
Description: Step Functions Orchestration

Resources:
  OrdersTable:
    Type: AWS::DynamoDB::Table
    Properties:
      TableName: Orders
      BillingMode: PAY_PER_REQUEST
      AttributeDefinitions: [{AttributeName: order_id, AttributeType: S}]
      KeySchema: [{AttributeName: order_id, KeyType: HASH}]

  # TODO: Add ReturnsTable DynamoDB resource with return_id as the key

  ValidateFn:
    Type: AWS::Serverless::Function
    Properties:
      CodeUri: handlers/
      Handler: validate.handler
      Runtime: python3.13
      Tracing: Active

  ChargeFn:
    Type: AWS::Serverless::Function
    Properties:
      CodeUri: handlers/
      Handler: charge.handler
      Runtime: python3.13
      Tracing: Active

  RecordFn:
    Type: AWS::Serverless::Function
    Properties:
      CodeUri: handlers/
      Handler: record.handler
      Runtime: python3.13
      Environment:
        Variables:
          TABLE: !Ref OrdersTable
      Policies:
        - DynamoDBCrudPolicy:
            TableName: !Ref OrdersTable
      Tracing: Active

  # TODO: Add CheckReturnPolicyFn Lambda function resource

  # TODO: Add ProcessRefundFn Lambda function resource

  # TODO: Add UpdateInventoryFn Lambda function resource with INVENTORY_TABLE environment variable and DynamoDB permissions

  OrderStateMachine:
    Type: AWS::Serverless::StateMachine
    Properties:
      Definition:
        Comment: Order flow
        StartAt: Validate
        States:
          Validate:
            Type: Task
            Resource: !GetAtt ValidateFn.Arn
            Next: Charge
          Charge:
            Type: Task
            Resource: !GetAtt ChargeFn.Arn
            Next: Record
          Record:
            Type: Task
            Resource: !GetAtt RecordFn.Arn
            End: true
      Policies:
        - LambdaInvokePolicy:
            FunctionName: !Ref ValidateFn
        - LambdaInvokePolicy:
            FunctionName: !Ref ChargeFn
        - LambdaInvokePolicy:
            FunctionName: !Ref RecordFn

  # TODO: Add ReturnStateMachine that orchestrates CheckReturnPolicy -> ProcessRefund -> UpdateInventory

Outputs:
  StateMachineArn:
    Value: !Ref OrderStateMachine
  # TODO: Add ReturnStateMachineArn output
```

This workflow mirrors the same three-role pattern as the order flow: `check_return_policy.py` plays the `validate.py` role (reads `return` from `event`, validates it, returns `{"return": return_data}`); `process_refund.py` plays the `charge.py` role (reads `return` from `event`, but attaches its new fields — `refund_id`, `refund_amount`, `refund_processed` — directly onto the **top-level** `event`, not nested inside `return`, then returns the whole `event`); and `update_inventory.py` plays the `record.py` role (reads `return_id`/`order_id`/`reason` from the nested `return`, but reads `refund_id`/`refund_amount` from the **top level** of the event, exactly like `record.py` reads `event["payment_id"]`).

**handlers/check_return_policy.py**

```python
def handler(event, context):
    return_data = event["return"]

    if return_data["days_since_purchase"] > 30:
        raise Exception("Return window expired")

    valid_reasons = ["defective", "wrong_item", "not_as_described"]
    if return_data["reason"] not in valid_reasons:
        raise Exception("Invalid return reason")

    return_data["policy_approved"] = True
    return {"return": return_data}
```

**handlers/process_refund.py**

```python
def handler(event, context):
    return_data = event["return"]
    event["refund_id"] = f"ref_{return_data['return_id']}"
    event["refund_amount"] = return_data["amount"]
    event["refund_processed"] = True
    return event
```

**handlers/update_inventory.py**

```python
import os, time, boto3
from decimal import Decimal
dynamodb = boto3.resource('dynamodb')
TABLE = os.environ.get("INVENTORY_TABLE", "Returns")

def handler(event, context):
    table = dynamodb.Table(TABLE)
    item = {
        "return_id": event["return"]["return_id"],
        "order_id": event["return"]["order_id"],
        "refund_id": event["refund_id"],
        "refund_amount": Decimal(str(event["refund_amount"])),
        "reason": event["return"]["reason"],
        "ts": int(time.time())
    }
    table.put_item(Item=item)
    return {"status": "ok", "return": item}
```

Here is the completed `template.yaml` with the `ReturnsTable`, the three new functions, the `ReturnStateMachine`, and its output added:

```yaml
AWSTemplateFormatVersion: '2010-09-09'
Transform: [AWS::Serverless-2016-10-31, AWS::LanguageExtensions]
Description: Step Functions Orchestration

Resources:
  OrdersTable:
    Type: AWS::DynamoDB::Table
    Properties:
      TableName: Orders
      BillingMode: PAY_PER_REQUEST
      AttributeDefinitions: [{AttributeName: order_id, AttributeType: S}]
      KeySchema: [{AttributeName: order_id, KeyType: HASH}]

  ReturnsTable:
    Type: AWS::DynamoDB::Table
    Properties:
      TableName: Returns
      BillingMode: PAY_PER_REQUEST
      AttributeDefinitions: [{AttributeName: return_id, AttributeType: S}]
      KeySchema: [{AttributeName: return_id, KeyType: HASH}]

  ValidateFn:
    Type: AWS::Serverless::Function
    Properties:
      CodeUri: handlers/
      Handler: validate.handler
      Runtime: python3.13
      Tracing: Active

  ChargeFn:
    Type: AWS::Serverless::Function
    Properties:
      CodeUri: handlers/
      Handler: charge.handler
      Runtime: python3.13
      Tracing: Active

  RecordFn:
    Type: AWS::Serverless::Function
    Properties:
      CodeUri: handlers/
      Handler: record.handler
      Runtime: python3.13
      Environment:
        Variables:
          TABLE: !Ref OrdersTable
      Policies:
        - DynamoDBCrudPolicy:
            TableName: !Ref OrdersTable
      Tracing: Active

  CheckReturnPolicyFn:
    Type: AWS::Serverless::Function
    Properties:
      CodeUri: handlers/
      Handler: check_return_policy.handler
      Runtime: python3.13
      Tracing: Active

  ProcessRefundFn:
    Type: AWS::Serverless::Function
    Properties:
      CodeUri: handlers/
      Handler: process_refund.handler
      Runtime: python3.13
      Tracing: Active

  UpdateInventoryFn:
    Type: AWS::Serverless::Function
    Properties:
      CodeUri: handlers/
      Handler: update_inventory.handler
      Runtime: python3.13
      Environment:
        Variables:
          INVENTORY_TABLE: !Ref ReturnsTable
      Policies:
        - DynamoDBCrudPolicy:
            TableName: !Ref ReturnsTable
      Tracing: Active

  OrderStateMachine:
    Type: AWS::Serverless::StateMachine
    Properties:
      Definition:
        Comment: Order flow
        StartAt: Validate
        States:
          Validate:
            Type: Task
            Resource: !GetAtt ValidateFn.Arn
            Next: Charge
          Charge:
            Type: Task
            Resource: !GetAtt ChargeFn.Arn
            Next: Record
          Record:
            Type: Task
            Resource: !GetAtt RecordFn.Arn
            End: true
      Policies:
        - LambdaInvokePolicy:
            FunctionName: !Ref ValidateFn
        - LambdaInvokePolicy:
            FunctionName: !Ref ChargeFn
        - LambdaInvokePolicy:
            FunctionName: !Ref RecordFn

  ReturnStateMachine:
    Type: AWS::Serverless::StateMachine
    Properties:
      Definition:
        Comment: Return flow
        StartAt: CheckReturnPolicy
        States:
          CheckReturnPolicy:
            Type: Task
            Resource: !GetAtt CheckReturnPolicyFn.Arn
            Next: ProcessRefund
          ProcessRefund:
            Type: Task
            Resource: !GetAtt ProcessRefundFn.Arn
            Next: UpdateInventory
          UpdateInventory:
            Type: Task
            Resource: !GetAtt UpdateInventoryFn.Arn
            End: true
      Policies:
        - LambdaInvokePolicy:
            FunctionName: !Ref CheckReturnPolicyFn
        - LambdaInvokePolicy:
            FunctionName: !Ref ProcessRefundFn
        - LambdaInvokePolicy:
            FunctionName: !Ref UpdateInventoryFn

Outputs:
  StateMachineArn:
    Value: !Ref OrderStateMachine
  ReturnStateMachineArn:
    Value: !Ref ReturnStateMachine
```